# EDA: Ecommerce Recommender
Исследование ориентировано на выбор memory-safe стратегии рекомендаций для задачи оптимизации `addtocart`.

## Ключевые выводы
- данные событийные и требуют `time-based split`
- основной позитивный сигнал: `addtocart`, более сильный дополнительный: `transaction`
- распределение активности длиннохвостое, поэтому система должна иметь fallback для короткой истории
- item properties полезны точечно, без полного расплющивания в wide table

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('..').resolve()
summary = json.loads((ROOT / 'eda_summary.json').read_text(encoding='utf-8'))
runtime_summary_path = ROOT / 'artifacts' / 'reports' / 'eda_runtime_summary.json'
runtime_summary = json.loads(runtime_summary_path.read_text(encoding='utf-8')) if runtime_summary_path.exists() else {}
summary, runtime_summary

In [ ]:
pd.DataFrame([
    {'metric': 'events_rows', 'value': summary['tables']['events_rows']},
    {'metric': 'users', 'value': summary['uniques']['users']},
    {'metric': 'items_in_events', 'value': summary['uniques']['items_in_events']},
    {'metric': 'item_properties_rows', 'value': summary['tables']['item_properties_rows']},
])

In [ ]:
event_share = pd.Series(summary['event_shares_pct']).sort_values(ascending=False)
plt.figure(figsize=(6, 4))
sns.barplot(x=event_share.index, y=event_share.values, palette='crest')
plt.title('Event share, %')
plt.xlabel('event')
plt.ylabel('share %')
plt.tight_layout()

## Интерпретация для моделирования
Высокая доля `view` при редких `addtocart` и `transaction` означает, что просмотры используются как контекст и сигнал для генерации кандидатов, но целевая оптимизация должна идти по событиям с более высокой ценностью.

In [ ]:
implications = pd.DataFrame({'implication': summary['implications']})
implications

## Memory-safe правила пайплайна
- загрузка только нужных колонок и downcasting типов
- отсутствие full join `events x item_properties`
- агрегация до уровня `user-item` перед построением item-to-item связей
- использование parquet для промежуточных артефактов

In [ ]:
pd.DataFrame(list(runtime_summary.items()), columns=['metric', 'value'])